# ⚙️ YT8M Genre Classifier — Stage 3: Preprocessing & DataLoader
Нормализация, сплиты, аугментация, WeightedSampler, DataLoader.

In [1]:
# ============================================================
# STAGE 3.0 — Импорты
# ============================================================
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle, json
from pathlib import Path

OUT_DIR   = Path(Path('../data2/processed'))
STAGE3_DIR = Path(Path('../data2/stage3'))
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

X_visual = np.load(OUT_DIR / 'X_visual.npy')
X_audio  = np.load(OUT_DIR / 'X_audio.npy')
y        = np.load(OUT_DIR / 'y.npy')
label_map = pd.read_csv(OUT_DIR / 'label_map.csv')
GENRES    = label_map.sort_values('label_idx')['category'].tolist()
N_CLASSES = len(GENRES)

print(f'✅ Загружено: X_visual={X_visual.shape}  X_audio={X_audio.shape}  y={y.shape}')
print(f'   Жанры ({N_CLASSES}): {GENRES}')


✅ Загружено: X_visual=(35636, 1024)  X_audio=(35636, 128)  y=(35636,)
   Жанры (12): ['Animals', 'Animation', 'Beauty', 'Dance', 'Film', 'Food', 'Gaming', 'Music', 'Performance', 'Sports', 'Tech', 'Vehicles']


In [2]:
# ============================================================
# STAGE 3.1 — Train / Val / Test split (70 / 15 / 15)
# ============================================================
idx = np.arange(len(y))

idx_train, idx_tmp = train_test_split(idx, test_size=0.30,
                                       stratify=y, random_state=42)
idx_val, idx_test  = train_test_split(idx_tmp, test_size=0.50,
                                       stratify=y[idx_tmp], random_state=42)

print('📊 Split:')
print(f'   Train : {len(idx_train):>6}  ({len(idx_train)/len(y)*100:.1f}%)')
print(f'   Val   : {len(idx_val):>6}  ({len(idx_val)/len(y)*100:.1f}%)')
print(f'   Test  : {len(idx_test):>6}  ({len(idx_test)/len(y)*100:.1f}%)')

# Проверяем что классы сохранились в каждом сплите
print('\n   Жанр           Train    Val   Test')
print('   ' + '-' * 40)
for i, genre in enumerate(GENRES):
    tr = (y[idx_train] == i).sum()
    va = (y[idx_val]   == i).sum()
    te = (y[idx_test]  == i).sum()
    print(f'   {genre:<14}  {tr:>5}  {va:>5}  {te:>5}')


📊 Split:
   Train :  24945  (70.0%)
   Val   :   5345  (15.0%)
   Test  :   5346  (15.0%)

   Жанр           Train    Val   Test
   ----------------------------------------
   Animals          2375    509    509
   Animation        2375    509    509
   Beauty            792    170    169
   Dance            2375    509    509
   Film             1171    250    251
   Food             2375    509    509
   Gaming           2375    509    509
   Music            2375    509    509
   Performance      2375    509    509
   Sports           2375    509    509
   Tech             1607    344    345
   Vehicles         2375    509    509


In [3]:
# ============================================================
# STAGE 3.2 — Нормализация (fit только на train)
# ============================================================
scaler_v = StandardScaler()
scaler_a = StandardScaler()

Xv_train = scaler_v.fit_transform(X_visual[idx_train])
Xv_val   = scaler_v.transform(X_visual[idx_val])
Xv_test  = scaler_v.transform(X_visual[idx_test])

Xa_train = scaler_a.fit_transform(X_audio[idx_train])
Xa_val   = scaler_a.transform(X_audio[idx_val])
Xa_test  = scaler_a.transform(X_audio[idx_test])

y_train = y[idx_train]
y_val   = y[idx_val]
y_test  = y[idx_test]

# Сохраняем скейлеры
with open(STAGE3_DIR / 'scalers.pkl', 'wb') as f:
    pickle.dump({'visual': scaler_v, 'audio': scaler_a}, f)

print('📐 После нормализации (train):')
print(f'  {"":<10}  {"mean":>10}  {"std":>10}  {"min":>10}  {"max":>10}')
print('  ' + '-' * 50)
for name, X in [("visual", Xv_train), ("audio", Xa_train)]:
    print(f'  {name:<10}  {X.mean():>10.4f}  {X.std():>10.4f}  '
          f'{X.min():>10.4f}  {X.max():>10.4f}')
print('✅ scalers.pkl сохранён')


📐 После нормализации (train):
                    mean         std         min         max
  --------------------------------------------------
  visual          0.0000      1.0000     -5.1752      5.2444
  audio          -0.0000      1.0000     -3.4200      3.4523
✅ scalers.pkl сохранён


In [4]:
# ============================================================
# STAGE 3.3 — Class weights для loss
# ============================================================
counts_train = np.bincount(y_train, minlength=N_CLASSES).astype(float)
class_weights = 1.0 / np.sqrt(counts_train)        # sqrt-inverse (мягче чем 1/n)
class_weights = class_weights / class_weights.sum() * N_CLASSES  # нормировка
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print('⚖️  Class weights (для FocalLoss/CrossEntropy):')
print(f'  {"Жанр":<14}  {"Train N":>8}  {"Weight":>8}')
print('  ' + '-' * 34)
for i, genre in enumerate(GENRES):
    print(f'  {genre:<14}  {int(counts_train[i]):>8}  {class_weights[i]:>8.4f}')

torch.save(class_weights_tensor, STAGE3_DIR / 'class_weights.pt')
np.save(STAGE3_DIR / 'class_weights.npy', class_weights)
print('\n✅ class_weights сохранён')


⚖️  Class weights (для FocalLoss/CrossEntropy):
  Жанр             Train N    Weight
  ----------------------------------
  Animals             2375    0.8974
  Animation           2375    0.8974
  Beauty               792    1.5541
  Dance               2375    0.8974
  Film                1171    1.2781
  Food                2375    0.8974
  Gaming              2375    0.8974
  Music               2375    0.8974
  Performance         2375    0.8974
  Sports              2375    0.8974
  Tech                1607    1.0910
  Vehicles            2375    0.8974

✅ class_weights сохранён


In [5]:
# ============================================================
# STAGE 3.4 — Dataset и DataLoader
# ============================================================
class VideoDataset(Dataset):
    """YT8M video-level dataset с опциональной аугментацией."""

    def __init__(self, X_visual, X_audio, y, augment=False):
        self.Xv  = torch.tensor(X_visual, dtype=torch.float32)
        self.Xa  = torch.tensor(X_audio,  dtype=torch.float32)
        self.y   = torch.tensor(y,        dtype=torch.long)
        self.aug = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        xv = self.Xv[idx].clone()
        xa = self.Xa[idx].clone()
        label = self.y[idx]

        if self.aug:
            # Gaussian noise
            xv = xv + torch.randn_like(xv) * 0.02
            xa = xa + torch.randn_like(xa) * 0.03
            # Random feature dropout (5%)
            if torch.rand(1).item() < 0.5:
                mask_v = torch.rand(xv.shape) > 0.05
                xv = xv * mask_v.float()

        return xv, xa, label


BATCH_SIZE  = 256   # YT8M больше данных → можно крупнее батч
NUM_WORKERS = 0

train_ds = VideoDataset(Xv_train, Xa_train, y_train, augment=True)
val_ds   = VideoDataset(Xv_val,   Xa_val,   y_val,   augment=False)
test_ds  = VideoDataset(Xv_test,  Xa_test,  y_test,  augment=False)

# WeightedRandomSampler только для train
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_ds),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                           sampler=sampler, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS)

print(f'✅ DataLoaders готовы:')
print(f'   BATCH_SIZE  = {BATCH_SIZE}')
print(f'   train steps = {len(train_loader)}')
print(f'   val   steps = {len(val_loader)}')
print(f'   test  steps = {len(test_loader)}')

# Проверяем один батч
xv_b, xa_b, y_b = next(iter(train_loader))
print(f'\n   Батч: xv={xv_b.shape}  xa={xa_b.shape}  y={y_b.shape}')
print(f'   dtype: xv={xv_b.dtype}  xa={xa_b.dtype}  y={y_b.dtype}')


✅ DataLoaders готовы:
   BATCH_SIZE  = 256
   train steps = 98
   val   steps = 21
   test  steps = 21

   Батч: xv=torch.Size([256, 1024])  xa=torch.Size([256, 128])  y=torch.Size([256])
   dtype: xv=torch.float32  xa=torch.float32  y=torch.int64


In [6]:
# ============================================================
# STAGE 3.5 — Сохранение
# ============================================================
np.save(STAGE3_DIR / 'Xv_train.npy', Xv_train)
np.save(STAGE3_DIR / 'Xv_val.npy',   Xv_val)
np.save(STAGE3_DIR / 'Xv_test.npy',  Xv_test)
np.save(STAGE3_DIR / 'Xa_train.npy', Xa_train)
np.save(STAGE3_DIR / 'Xa_val.npy',   Xa_val)
np.save(STAGE3_DIR / 'Xa_test.npy',  Xa_test)
np.save(STAGE3_DIR / 'y_train.npy',  y_train)
np.save(STAGE3_DIR / 'y_val.npy',    y_val)
np.save(STAGE3_DIR / 'y_test.npy',   y_test)
np.save(STAGE3_DIR / 'idx_train.npy', idx_train)
np.save(STAGE3_DIR / 'idx_val.npy',   idx_val)
np.save(STAGE3_DIR / 'idx_test.npy',  idx_test)

config = {
    'n_train'    : int(len(y_train)),
    'n_val'      : int(len(y_val)),
    'n_test'     : int(len(y_test)),
    'n_classes'  : N_CLASSES,
    'genres'     : GENRES,
    'dim_visual' : int(Xv_train.shape[1]),
    'dim_audio'  : int(Xa_train.shape[1]),
    'batch_size' : BATCH_SIZE,
    'class_weights': class_weights.tolist(),
}
with open(STAGE3_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('=' * 55)
print('       STAGE 3 — COMPLETE')
print('=' * 55)
print(f'  Train  : {len(y_train):,}')
print(f'  Val    : {len(y_val):,}')
print(f'  Test   : {len(y_test):,}')
print(f'  Visual : {Xv_train.shape[1]} dims')
print(f'  Audio  : {Xa_train.shape[1]} dims')
print(f'  Batch  : {BATCH_SIZE}')
print(f'\n  Файлы в {STAGE3_DIR}:')
total_mb = 0
for f in sorted(STAGE3_DIR.iterdir()):
    mb = f.stat().st_size / 1024**2
    total_mb += mb
    print(f'   {f.name:<25} {mb:.1f} MB')
print(f'   {"ИТОГО":<25} {total_mb:.1f} MB')
print('=' * 55)
print('✅ Готово к Stage 4 — Model Architecture')


       STAGE 3 — COMPLETE
  Train  : 24,945
  Val    : 5,345
  Test   : 5,346
  Visual : 1024 dims
  Audio  : 128 dims
  Batch  : 256

  Файлы в data2/stage3:
   Xa_test.npy               2.6 MB
   Xa_train.npy              12.2 MB
   Xa_val.npy                2.6 MB
   Xv_test.npy               20.9 MB
   Xv_train.npy              97.4 MB
   Xv_val.npy                20.9 MB
   class_weights.npy         0.0 MB
   class_weights.pt          0.0 MB
   config.json               0.0 MB
   idx_test.npy              0.0 MB
   idx_train.npy             0.2 MB
   idx_val.npy               0.0 MB
   scalers.pkl               0.0 MB
   y_test.npy                0.0 MB
   y_train.npy               0.2 MB
   y_val.npy                 0.0 MB
   ИТОГО                     157.2 MB
✅ Готово к Stage 4 — Model Architecture
